# 第3章 BLE
アドバタイジングパケットを復調します。実測データ（`ble_ch37.iq`）がなければ、テスト用のIQを生成して使います。

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, "..")  # code/ 直下のモジュールを読み込む
DATA = Path(os.environ.get("SDR_BOOK_DATA", "../data"))  # IQ データの置き場所
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import ble

path = DATA / "ble_ch37.iq"
if not path.exists():
    path = Path("ble_test.iq")
    ble.make_test_iq(path)
x = ble.load_iq(path)
bursts = ble.detect_bursts(x)
print(path, "bursts:", len(bursts))

## 周波数（位相差）の波形

In [ ]:
s, e = bursts[0]
freq = ble.fm_demod(x[s:e])
plt.figure(figsize=(9, 3)); plt.plot(freq[:800]); plt.ylabel("phase diff [rad]"); plt.show()

## ビット列 → アクセスアドレス → デホワイトニング → CRC

In [ ]:
for s, e in bursts:
    bits = ble.slice_bits(ble.fm_demod(x[s:e]))
    pos = ble.find_access_address(bits)
    if pos is None:
        continue
    r = ble.read_packet(bits, pos, ble.CHANNEL)
    if r is None:
        continue
    pdu, crc = r
    ok = ble.ble_crc(pdu) == crc
    print(f"t={s / ble.SAMPLE_RATE * 1e3:9.3f}ms CRC {'OK' if ok else 'NG'}")
    if ok:
        adv = ble.parse_pdu(pdu)
        print("   ", adv["type"], adv["adv_addr"])
        for name, value in adv["ad"]:
            print("      ", name, ble.format_ad(name, value))